# 第六节：守恒量约束 PINN 求解粘性 Burgers 方程

> 本 Notebook 已将长讲义归纳为“问题、方法、实验、结论”四条主线，并包含完整可运行代码。

## 学习目标

1. 从线性平流迁移到非线性 Burgers 方程；
2. 理解质量守恒约束如何加入 PINN；
3. 比较 Baseline PINN 与 Conservation-PINN；
4. 检查训练区间与长时间外推区间的误差和质量漂移。

## Apple Silicon 运行说明

- 第六节优先使用 MacBook 的 `mps`，不可用时自动回退到 CUDA 或 CPU；
- MPS 使用 `float32`，适合 PINN 训练；
- 如果当前 PyTorch 的 MPS 高阶自动微分报错，可设置 `cfg.device = "cpu"`；
- 首次运行保留最后一个单元中的 `RUN_QUICK = True`。


# 1. 数学问题

粘性 Burgers 方程为：

$$u_t + u u_x = \nu u_{xx}, \qquad x\in[0,1],\ t\in[0,2]$$

也可写成守恒律形式：

$$u_t+\left(\frac{u^2}{2}\right)_x=\nu u_{xx}$$

初值与周期边界为：

$$u(x,0)=1+0.5\sin(2\pi x)$$

$$u(0,t)=u(1,t),\qquad u_x(0,t)=u_x(1,t)$$

在周期边界下，总质量保持不变：

$$M(t)=\int_0^1u(x,t)\,dx=M(0)=1$$


# 2. PINN 与守恒约束

网络预测记为 $u_\theta(x,t)$，PDE 残差为：

$$f_\theta=u_t+u_\theta u_x-\nu u_{xx}$$

基础损失为：

$$\mathcal L_{\mathrm{base}}
=\lambda_{ic}\mathcal L_{ic}
+\lambda_{bc}\mathcal L_{bc}
+\lambda_f\mathcal L_f$$

守恒量损失为：

$$\mathcal L_{\mathrm{cons}}
=\frac{1}{N_t}\sum_{n=1}^{N_t}
\left|
\frac{1}{N_x}\sum_{k=1}^{N_x}u_\theta(x_k,t_n)-1
\right|^2$$

Conservation-PINN 的总损失为：

$$\mathcal L
=\mathcal L_{\mathrm{base}}
+\lambda_{\mathrm{cons}}\mathcal L_{\mathrm{cons}}$$


# 3. 公平对比实验

| 项目 | Baseline PINN | Conservation-PINN |
|---|---:|---:|
| 网络结构 | 相同 | 相同 |
| IC / BC / PDE 点 | 相同 | 相同 |
| 优化器和随机种子 | 相同 | 相同 |
| 守恒权重 | $0$ | $\lambda_{\mathrm{cons}}>0$ |

训练区间为 $t\in[0,1]$，评估延伸到 $t\in[0,2]$。重点比较：

$$E_{L2}=\frac{\|u_\theta-u_{\mathrm{ref}}\|_2}{\|u_{\mathrm{ref}}\|_2}$$

$$E_{\mathrm{cons}}(t)=|M_\theta(t)-1|$$

守恒误差更小不代表所有点的误差一定更小，因此必须同时看解场、时间切片、相对误差和质量漂移。


# 4. 实验流程

1. 用周期伪谱法生成参考解；
2. 训练 Baseline PINN；
3. 训练 Conservation-PINN；
4. 输出训练曲线、解场误差、质量漂移、时间切片和 CSV 指标；
5. 正式实验可比较 $\lambda_{\mathrm{cons}}=0.1,1,10$。


# 配套实验代码

下面按实验流程排列完整代码。首次运行保留 `RUN_QUICK = True`，确认流程正确后再运行正式实验。


## 导入库、参数与 Apple Silicon 设备选择


In [ ]:
from __future__ import annotations

import argparse
import csv
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


@dataclass
class Config:
    nu: float = 0.01 / np.pi
    train_t_max: float = 1.0
    test_t_max: float = 2.0
    epochs: int = 3000
    learning_rate: float = 1e-3
    n_ic: int = 128
    n_bc: int = 128
    n_f: int = 3000
    n_cons_t: int = 50
    n_cons_x: int = 200
    hidden_dim: int = 64
    num_hidden: int = 4
    lambda_ic: float = 10.0
    lambda_bc: float = 1.0
    lambda_f: float = 1.0
    lambda_cons: float = 1.0
    seed: int = 42
    print_every: int = 100
    ref_nx: int = 256
    ref_dt: float = 5e-4
    device: str = "auto"
    output_dir: str = "results/section6_burgers"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_device(preference: str = "auto") -> torch.device:
    """Resolve auto/mps/cuda/cpu while giving useful availability errors."""
    preference = preference.lower()
    if preference not in {"auto", "mps", "cuda", "cpu"}:
        raise ValueError("device must be one of: auto, mps, cuda, cpu")
    if preference == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS was requested but is not available in this PyTorch build")
    if preference == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is not available")
    if preference != "auto":
        return torch.device(preference)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def initial_condition(x):
    """Smooth periodic initial condition with exact mass M0 = 1."""
    if isinstance(x, torch.Tensor):
        return 1.0 + 0.5 * torch.sin(2.0 * torch.pi * x)
    return 1.0 + 0.5 * np.sin(2.0 * np.pi * x)


## 周期伪谱参考解


In [ ]:
def solve_reference(
    times: np.ndarray, nu: float, nx: int = 256, dt: float = 5e-4
) -> tuple[np.ndarray, np.ndarray]:
    """Solve viscous Burgers on [0, 1) using spectral derivatives and RK4."""
    times = np.asarray(times, dtype=np.float64)
    if times.ndim != 1 or np.any(np.diff(times) < 0):
        raise ValueError("times must be a sorted one-dimensional array")

    x = np.arange(nx, dtype=np.float64) / nx
    wave_numbers = 2.0 * np.pi * np.fft.fftfreq(nx, d=1.0 / nx)
    cutoff = nx // 3
    dealias_mask = np.abs(np.fft.fftfreq(nx) * nx) <= cutoff
    u = initial_condition(x)

    def rhs(values: np.ndarray) -> np.ndarray:
        u_hat = np.fft.fft(values)
        flux_hat = np.fft.fft(0.5 * values**2)
        flux_hat[~dealias_mask] = 0.0
        flux_x = np.fft.ifft(1j * wave_numbers * flux_hat).real
        u_xx = np.fft.ifft(-(wave_numbers**2) * u_hat).real
        return -flux_x + nu * u_xx

    snapshots = np.empty((len(times), nx), dtype=np.float64)
    current_t = 0.0
    for index, target_t in enumerate(times):
        while current_t < target_t - 1e-14:
            step = min(dt, target_t - current_t)
            k1 = rhs(u)
            k2 = rhs(u + 0.5 * step * k1)
            k3 = rhs(u + 0.5 * step * k2)
            k4 = rhs(u + step * k3)
            u = u + step * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
            current_t += step
        snapshots[index] = u
    return x, snapshots


## PINN 网络与自动微分


In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden_dim: int = 64, num_hidden: int = 4):
        super().__init__()
        layers: list[nn.Module] = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(num_hidden - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.Tanh()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, t], dim=1))


def gradient(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return torch.autograd.grad(
        y,
        x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]


def burgers_residual(
    model: PINN, x: torch.Tensor, t: torch.Tensor, nu: float
) -> torch.Tensor:
    """f = u_t + u u_x - nu u_xx."""
    u = model(x, t)
    u_t = gradient(u, t)
    u_x = gradient(u, x)
    u_xx = gradient(u_x, x)
    return u_t + u * u_x - nu * u_xx


def periodic_bc_loss(model: PINN, t: torch.Tensor) -> torch.Tensor:
    """Enforce both u(0,t)=u(1,t) and u_x(0,t)=u_x(1,t)."""
    x_left = torch.zeros_like(t, requires_grad=True)
    x_right = torch.ones_like(t, requires_grad=True)
    u_left = model(x_left, t)
    u_right = model(x_right, t)
    ux_left = gradient(u_left, x_left)
    ux_right = gradient(u_right, x_right)
    return torch.mean((u_left - u_right) ** 2) + torch.mean(
        (ux_left - ux_right) ** 2
    )


def conservation_loss(
    model: PINN, times: torch.Tensor, n_x: int, target_mass: float = 1.0
) -> torch.Tensor:
    """Approximate M(t)=integral_0^1 u(x,t) dx on uniform periodic points."""
    x = torch.arange(n_x, device=times.device, dtype=times.dtype) / n_x
    x = x.reshape(1, -1).expand(len(times), -1).reshape(-1, 1)
    t = times.reshape(-1, 1).expand(-1, n_x).reshape(-1, 1)
    mass = model(x, t).reshape(len(times), n_x).mean(dim=1)
    return torch.mean((mass - target_mass) ** 2)


## 统一训练函数


In [ ]:
def train_model(
    name: str, config: Config, device: torch.device, lambda_cons: float
) -> tuple[PINN, dict[str, list[float]]]:
    set_seed(config.seed)
    model = PINN(config.hidden_dim, config.num_hidden).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    history = {key: [] for key in ["total", "ic", "bc", "pde", "cons"]}

    for epoch in range(1, config.epochs + 1):
        optimizer.zero_grad()

        x_ic = torch.rand(config.n_ic, 1, device=device)
        t_ic = torch.zeros_like(x_ic)
        loss_ic = torch.mean((model(x_ic, t_ic) - initial_condition(x_ic)) ** 2)

        t_bc = config.train_t_max * torch.rand(config.n_bc, 1, device=device)
        loss_bc = periodic_bc_loss(model, t_bc)

        x_f = torch.rand(config.n_f, 1, device=device, requires_grad=True)
        t_f = (
            config.train_t_max
            * torch.rand(config.n_f, 1, device=device, requires_grad=True)
        )
        residual = burgers_residual(model, x_f, t_f, config.nu)
        loss_pde = torch.mean(residual**2)

        t_cons = config.train_t_max * torch.rand(config.n_cons_t, 1, device=device)
        if lambda_cons > 0.0:
            loss_cons = conservation_loss(model, t_cons, config.n_cons_x)
        else:
            # Keep a diagnostic value for the baseline without building its
            # conservation-loss graph into backpropagation.
            with torch.no_grad():
                loss_cons = conservation_loss(model, t_cons, config.n_cons_x)

        loss = (
            config.lambda_ic * loss_ic
            + config.lambda_bc * loss_bc
            + config.lambda_f * loss_pde
            + lambda_cons * loss_cons
        )
        loss.backward()
        optimizer.step()

        values = [loss, loss_ic, loss_bc, loss_pde, loss_cons]
        for key, value in zip(history, values):
            history[key].append(float(value.detach().cpu()))

        if epoch == 1 or epoch % config.print_every == 0:
            print(
                f"{name:>13s} | epoch {epoch:5d}/{config.epochs} | "
                f"loss={history['total'][-1]:.3e} | "
                f"pde={history['pde'][-1]:.3e} | "
                f"cons={history['cons'][-1]:.3e}"
            )
    return model, history


## 评估指标


In [ ]:
@torch.no_grad()
def predict(
    model: PINN, x: np.ndarray, times: np.ndarray, device: torch.device
) -> np.ndarray:
    xx, tt = np.meshgrid(x, times)
    inputs_x = torch.tensor(xx.reshape(-1, 1), dtype=torch.float32, device=device)
    inputs_t = torch.tensor(tt.reshape(-1, 1), dtype=torch.float32, device=device)
    values = model(inputs_x, inputs_t).cpu().numpy()
    return values.reshape(len(times), len(x))


def metrics(prediction: np.ndarray, reference: np.ndarray) -> dict[str, float]:
    mass_error = np.abs(prediction.mean(axis=1) - 1.0)
    return {
        "relative_l2": float(np.linalg.norm(prediction - reference) / np.linalg.norm(reference)),
        "mean_cons_error": float(mass_error.mean()),
        "max_cons_error": float(mass_error.max()),
        "final_cons_error": float(mass_error[-1]),
    }


def save_metrics(rows: list[dict[str, float | str]], output_dir: Path) -> None:
    fieldnames = [
        "method",
        "relative_l2",
        "mean_cons_error",
        "max_cons_error",
        "final_cons_error",
    ]
    with (output_dir / "metrics.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


## 结果可视化


In [ ]:
def plot_training(
    histories: dict[str, dict[str, list[float]]], output_dir: Path
) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, history in histories.items():
        axes[0].semilogy(history["total"], label=name)
        axes[1].semilogy(history["cons"], label=name)
    axes[0].set(title="Total training loss", xlabel="epoch", ylabel="loss")
    axes[1].set(title="Mass loss during training", xlabel="epoch", ylabel="loss")
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "01_training_loss.png", dpi=180)
    plt.close(fig)


def plot_solution_fields(
    x: np.ndarray,
    times: np.ndarray,
    reference: np.ndarray,
    predictions: dict[str, np.ndarray],
    output_dir: Path,
) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
    extent = [x[0], x[-1], times[-1], times[0]]
    fields = [reference, predictions["Baseline"], predictions["Conservation"]]
    titles = ["Reference", "Baseline PINN", "Conservation-PINN"]
    vmax = max(np.max(np.abs(field - reference)) for field in fields[1:])
    for column, (field, title) in enumerate(zip(fields, titles)):
        image = axes[0, column].imshow(field, aspect="auto", extent=extent, cmap="viridis")
        axes[0, column].set_title(title)
        fig.colorbar(image, ax=axes[0, column])
        error = np.abs(field - reference)
        image = axes[1, column].imshow(
            error, aspect="auto", extent=extent, cmap="magma", vmin=0.0, vmax=vmax
        )
        axes[1, column].set_title(f"{title} absolute error")
        fig.colorbar(image, ax=axes[1, column])
    for axis in axes[:, 0]:
        axis.set_ylabel("t")
    for axis in axes[-1]:
        axis.set_xlabel("x")
    fig.tight_layout()
    fig.savefig(output_dir / "02_solution_and_error.png", dpi=180)
    plt.close(fig)


def plot_mass_drift(
    times: np.ndarray, predictions: dict[str, np.ndarray], output_dir: Path
) -> None:
    fig, axis = plt.subplots(figsize=(7, 4))
    for name, prediction in predictions.items():
        axis.semilogy(times, np.abs(prediction.mean(axis=1) - 1.0) + 1e-16, label=name)
    axis.axvline(1.0, color="black", linestyle="--", linewidth=1, label="training limit")
    axis.set(xlabel="t", ylabel="|M(t) - M(0)|", title="Mass conservation error")
    axis.grid(alpha=0.3)
    axis.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "03_mass_drift.png", dpi=180)
    plt.close(fig)


def plot_time_slices(
    x: np.ndarray,
    times: np.ndarray,
    reference: np.ndarray,
    predictions: dict[str, np.ndarray],
    output_dir: Path,
) -> None:
    requested = [0.0, 0.5, 1.0, 1.5, 2.0]
    indices = [int(np.argmin(np.abs(times - value))) for value in requested]
    fig, axes = plt.subplots(1, len(indices), figsize=(18, 3.5), sharey=True)
    for axis, index in zip(axes, indices):
        axis.plot(x, reference[index], color="black", linewidth=2, label="Reference")
        for name, prediction in predictions.items():
            axis.plot(x, prediction[index], linestyle="--", label=name)
        axis.set_title(f"t = {times[index]:.1f}")
        axis.set_xlabel("x")
        axis.grid(alpha=0.3)
    axes[0].set_ylabel("u(x,t)")
    axes[-1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / "04_time_slices.png", dpi=180)
    plt.close(fig)


## 完整对比实验


In [ ]:
def run_experiment(config: Config) -> None:
    set_seed(config.seed)
    device = select_device(config.device)
    output_dir = Path.cwd() / config.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"device: {device}")
    print(f"results: {output_dir}")

    baseline, baseline_history = train_model("Baseline", config, device, lambda_cons=0.0)
    conservation, conservation_history = train_model(
        "Conservation", config, device, lambda_cons=config.lambda_cons
    )

    times = np.linspace(0.0, config.test_t_max, 101)
    x, reference = solve_reference(times, config.nu, config.ref_nx, config.ref_dt)
    predictions = {
        "Baseline": predict(baseline, x, times, device),
        "Conservation": predict(conservation, x, times, device),
    }

    rows: list[dict[str, float | str]] = []
    for name, prediction in predictions.items():
        row: dict[str, float | str] = {"method": name}
        row.update(metrics(prediction, reference))
        rows.append(row)
        print(name, {key: f"{value:.3e}" for key, value in row.items() if key != "method"})

    save_metrics(rows, output_dir)
    plot_training(
        {"Baseline": baseline_history, "Conservation": conservation_history}, output_dir
    )
    plot_solution_fields(x, times, reference, predictions, output_dir)
    plot_mass_drift(times, predictions, output_dir)
    plot_time_slices(x, times, reference, predictions, output_dir)
    torch.save(baseline.state_dict(), output_dir / "baseline_pinn.pt")
    torch.save(conservation.state_dict(), output_dir / "conservation_pinn.pt")


## 运行实验


In [ ]:
RUN_QUICK = True

cfg = Config()
if RUN_QUICK:
    cfg.epochs = 10
    cfg.n_ic = 32
    cfg.n_bc = 32
    cfg.n_f = 128
    cfg.n_cons_t = 8
    cfg.n_cons_x = 32
    cfg.hidden_dim = 32
    cfg.num_hidden = 3
    cfg.print_every = 1
    cfg.ref_nx = 128
    cfg.ref_dt = 1e-3
    cfg.output_dir = "results/section6_burgers_quick"

run_experiment(cfg)
